In [1]:
!nvidia-smi

Mon Nov 23 13:06:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [3]:
%%writefile matrix_mul.cu

Writing matrix_mul.cu


In [4]:
%%writefile matrix_mul.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cuda.h>
#include <sys/time.h>
#include <stdlib.h>
#include <stdbool.h>

#ifndef _COMMON_H
#define _COMMON_H

#define CHECK(call) \
{ \
    const cudaError_t error = call; \
    if (error != cudaSuccess) \
    { \
        fprintf(stderr, "Error: %s:%d, ", __FILE__, __LINE__); \
        fprintf(stderr, "code: %d, reason: %s\n", error, cudaGetErrorString(error)); \
        exit(1); \
    } \
}

inline double seconds()
{
    struct timeval tp;
    struct timezone tzp;
    gettimeofday(&tp, &tzp);
    return ((double)tp.tv_sec + (double)tp.tv_usec * 1.e-6);
}

#endif

#define SIZE 4
#define BLOCK_SIZE 2

void matrixMultiplyHost(int *a, int *b, int *c, int size)
{
    for (int row = 0; row < size; ++row)
    {
        for (int col = 0; col < size; ++col)
        {
            int sum = 0;
            for (int k = 0; k < size; ++k)
            {
                sum += a[row * size + k] * b[k * size + col];
            }
            c[row * size + col] = sum;
        }
    }
}

__global__ void matrixMultiply(int *a, int *b, int *c, int size)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < size && col < size)
    {
        int sum = 0;
        for (int k = 0; k < size; ++k)
        {
            sum += a[row * size + k] * b[k * size + col];
        }
        c[row * size + col] = sum;
    }
}

int main()
{
    int a[SIZE * SIZE], b[SIZE * SIZE], c[SIZE * SIZE], c_host[SIZE * SIZE];
    int *dev_a, *dev_b, *dev_c;
    int bytes = SIZE * SIZE * sizeof(int);

    for (int i = 0; i < SIZE; ++i)
    {
        for (int j = 0; j < SIZE; ++j)
        {
            a[i * SIZE + j] = i + 1;
            b[i * SIZE + j] = j + 1;
        }
    }

    CHECK(cudaMalloc((void**)&dev_a, bytes));
    CHECK(cudaMalloc((void**)&dev_b, bytes));
    CHECK(cudaMalloc((void**)&dev_c, bytes));

    CHECK(cudaMemcpy(dev_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dev_b, b, bytes, cudaMemcpyHostToDevice));

    dim3 dimBlock(BLOCK_SIZE, BLOCK_SIZE);
    dim3 dimGrid((SIZE + BLOCK_SIZE - 1) / BLOCK_SIZE,
                 (SIZE + BLOCK_SIZE - 1) / BLOCK_SIZE);

    struct timeval start, end;

    gettimeofday(&start, NULL);

    matrixMultiply<<<dimGrid, dimBlock>>>(dev_a, dev_b, dev_c, SIZE);

    CHECK(cudaDeviceSynchronize());
    CHECK(cudaGetLastError());

    gettimeofday(&end, NULL);

    double elapsed_gpu =
        (end.tv_sec - start.tv_sec) +
        (end.tv_usec - start.tv_usec) / 1e6;

    CHECK(cudaMemcpy(c, dev_c, bytes, cudaMemcpyDeviceToHost));

    gettimeofday(&start, NULL);

    matrixMultiplyHost(a, b, c_host, SIZE);

    gettimeofday(&end, NULL);

    double elapsed_cpu =
        (end.tv_sec - start.tv_sec) +
        (end.tv_usec - start.tv_usec) / 1e6;

    printf("Result Matrix from GPU:\n");

    for (int i = 0; i < SIZE; ++i)
    {
        for (int j = 0; j < SIZE; ++j)
        {
            printf("%d ", c[i * SIZE + j]);
        }
        printf("\n");
    }

    printf("\nResult Matrix from CPU:\n");

    for (int i = 0; i < SIZE; ++i)
    {
        for (int j = 0; j < SIZE; ++j)
        {
            printf("%d ", c_host[i * SIZE + j]);
        }
        printf("\n");
    }

    printf("\nGPU Time: %.6f s\n", elapsed_gpu);
    printf("CPU Time: %.6f s\n", elapsed_cpu);

    CHECK(cudaFree(dev_a));
    CHECK(cudaFree(dev_b));
    CHECK(cudaFree(dev_c));

    return 0;
}

Overwriting matrix_mul.cu


In [5]:
!nvcc -arch=sm_75 -o matrix_mul matrix_mul.cu

In [6]:
!./matrix_mul

Result Matrix from GPU:
4 8 12 16 
8 16 24 32 
12 24 36 48 
16 32 48 64 

Result Matrix from CPU:
4 8 12 16 
8 16 24 32 
12 24 36 48 
16 32 48 64 

GPU Time: 0.000660 s
CPU Time: 0.000001 s
